# 第14・15回：CNNとTransformerの基礎 — MNIST画像分類実習

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session14_15_cnn_llm_mnist.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

> この教材では、質問文の「LMM」を **LLM（Large Language Model：大規模言語モデル）** の意味として扱います。

CNNとTransformerは、どちらも深層学習の代表的なアーキテクチャです。CNNは小さなフィルタで局所的な模様を探し、TransformerはSelf-Attentionで入力要素同士の関係を調べます。LLMは、主にTransformerを大規模な文章データで学習したモデルです。

第14回ではパーセプトロン、活性化関数、CNNの仕組みを整理します。第15回ではTransformerとLLMの関係を学んだ後、MNISTを用いてCNNを構築・学習・評価します。

---

## 📋 到達目標

1. パーセプトロンと多層パーセプトロンの違いを説明できる
2. 活性化関数が非線形な分類に必要な理由を説明できる
3. バックプロパゲーションとオプティマイザの役割を区別できる
4. Dropoutの目的とdrop rateの意味を説明できる
5. 畳み込みとプーリングの役割を説明できる
6. CNNが画像認識に適している理由を説明できる
7. Transformerのトークン、埋め込み、Self-Attentionを説明できる
8. CNNとTransformerの共通点・相違点を整理できる
9. TransformerとLLMの関係、次トークン予測を説明できる
10. KerasでMNIST分類用CNNを実装し、学習曲線や混同行列から評価できる

---

# 第14回：パーセプトロン・活性化関数・CNNの基礎

## 1. 深層学習に共通する考え方

### 1-1. パーセプトロン：人工ニューロンの基本単位

パーセプトロンは、複数の入力を受け取り、1つの出力を返す最も基本的な人工ニューロンです。それぞれの入力に重みを掛けて合計し、バイアスを加えた後、活性化関数へ渡します。

![パーセプトロンの構造](images/session14_15/perceptron.png)

計算は次式で表せます。

$$
z = \sum_i w_i x_i + b, \qquad a = f(z)
$$

- $x_i$：入力
- $w_i$：入力の重要度を表す重み
- $b$：バイアス
- $f$：ReLUなどの活性化関数
- $a$：次の層に渡す出力

初期のパーセプトロンでは、加重和が基準以上なら1、未満なら0を返すステップ関数が使われました。これは直線や平面で分けられる問題には対応できますが、XORのように1本の直線では分けられない問題を学習できません。

### 1-2. 多層パーセプトロン（MLP）

複数のパーセプトロンを層として並べたものが**多層パーセプトロン（Multi-Layer Perceptron: MLP）**です。

![多層パーセプトロンの構造](images/session14_15/mlp.png)

各矢印には学習によって更新される重みがあります。各隠れユニットでは「加重和 → 活性化関数」を計算し、その出力を次の層へ渡します。

- **入力層**：元データを受け取る
- **隠れ層**：入力を段階的に変換し、分類に役立つ特徴を作る
- **出力層**：目的に合わせた予測値やクラス確率を返す

`Dense` は、前の層のすべてのユニットと接続する全結合層です。層を重ねることで複雑な表現を作れますが、そのためには各層に活性化関数を入れる必要があります。

### 1-3. 活性化関数はなぜ必要か

活性化関数は、加重和を次の層へどのように伝えるかを決める関数です。ニューロンが「どの入力に、どの程度反応するか」を表します。

活性化関数を使わず、線形変換だけを何層重ねても、全体は結局1回の線形変換と同じです。

$$
W_2(W_1x + b_1) + b_2 = (W_2W_1)x + (W_2b_1 + b_2)
$$

そこで、層の間に**非線形な活性化関数**を入れます。これにより、曲線や複雑な領域でクラスを分けられるようになります。

| 活性化関数 | 出力範囲 | 主な用途・特徴 |
| --- | --- | --- |
| ステップ関数 | 0または1 | 初期のパーセプトロン。微分できないため、現在の深層学習の学習には通常使わない |
| Sigmoid | 0〜1 | 二値分類の出力層。値を確率として解釈しやすい |
| tanh | -1〜1 | 0を中心とした出力。古典的な隠れ層や一部の系列モデルで使用 |
| ReLU | 0以上 | 隠れ層で広く使用。計算が単純で勾配消失を緩和しやすい |
| Softmax | 各値が0〜1、合計1 | 多クラス分類の出力層。クラスごとの確率を表す |

#### ReLU

ReLU（Rectified Linear Unit）は、負の入力を0にし、正の入力をそのまま通します。

$$
\mathrm{ReLU}(x) = \max(0, x)
$$

今回のCNNでは、2つの`Conv2D`層にReLUを使います。画像中の特徴に強く反応した正の値を残しながら、モデルに非線形性を加えます。最後の`Dense(10)`層には、多クラス分類用のSoftmaxを使います。

#### SigmoidとSoftmax

Sigmoidは1つの値を0〜1へ変換するため、「該当する／しない」のような二値分類の出力に向いています。Softmaxは複数の値を合計1の確率へ変換するため、0〜9を選ぶMNISTのような多クラス分類に向いています。

```text
二値分類：     Dense(1, activation="sigmoid")
10クラス分類：Dense(10, activation="softmax")
```

> 活性化関数は目的に合わせて選びます。特に出力層の活性化関数、正解ラベルの形式、損失関数の組み合わせが重要です。

### 1-4. 演習：隠れ層と活性化関数が分類境界に与える影響

直線では分けにくい半月状のデータを使い、次の3モデルを順番に確認します。

- **モデルA-1**：活性化関数なし、隠れ層2層
- **モデルA-2**：活性化関数なし、隠れ層4層
- **モデルB**：ReLUあり、隠れ層2層

まずモデルA-1とA-2を比べ、活性化関数がなければ層を増やしても分類境界が線形のままであることを確認します。その後、モデルBの非線形な分類境界を単独で表示します。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
keras.utils.set_random_seed(SEED)

X_moons, y_moons = make_moons(
    n_samples=1000,
    noise=0.20,
    random_state=SEED,
)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons,
    y_moons,
    test_size=0.25,
    stratify=y_moons,
    random_state=SEED,
)

scaler_m = StandardScaler()
X_train_m = scaler_m.fit_transform(X_train_m)
X_test_m = scaler_m.transform(X_test_m)

plt.figure(figsize=(6, 5))
plt.scatter(
    X_train_m[:, 0], X_train_m[:, 1],
    c=y_train_m, cmap="coolwarm", s=20, alpha=0.8,
)
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.title("Nonlinear two-class data: make_moons")
plt.show()


#### 共通関数：隠れ層数を指定してMLPを作る


In [ ]:
def build_mlp(hidden_layer_count, hidden_activation, model_name):
    model = keras.Sequential(name=model_name)
    model.add(layers.Input(shape=(2,)))

    for _ in range(hidden_layer_count):
        model.add(layers.Dense(8, activation=hidden_activation))

    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.01),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


def train_and_evaluate_mlp(model):
    history = model.fit(
        X_train_m,
        y_train_m,
        validation_split=0.2,
        epochs=100,
        batch_size=32,
        verbose=0,
    )
    test_loss, test_accuracy = model.evaluate(
        X_test_m, y_test_m, verbose=0
    )
    return history, test_loss, test_accuracy


#### モデルA-1：活性化関数なし・隠れ層2層


In [ ]:
keras.utils.set_random_seed(SEED)

linear_model_2 = build_mlp(
    hidden_layer_count=2,
    hidden_activation=None,
    model_name="linear_2_hidden_layers",
)

history_a1, loss_a1, accuracy_a1 = train_and_evaluate_mlp(
    linear_model_2
)

print(f"A-1（隠れ層2層）のテスト正解率: {accuracy_a1:.3f}")


#### モデルA-2：活性化関数なし・隠れ層4層


In [ ]:
keras.utils.set_random_seed(SEED)

linear_model_4 = build_mlp(
    hidden_layer_count=4,
    hidden_activation=None,
    model_name="linear_4_hidden_layers",
)

history_a2, loss_a2, accuracy_a2 = train_and_evaluate_mlp(
    linear_model_4
)

print(f"A-2（隠れ層4層）のテスト正解率: {accuracy_a2:.3f}")

comparison_a = pd.DataFrame({
    "model": ["A-1: 2 hidden layers", "A-2: 4 hidden layers"],
    "hidden_activation": ["None", "None"],
    "parameters": [
        linear_model_2.count_params(),
        linear_model_4.count_params(),
    ],
    "test_loss": [loss_a1, loss_a2],
    "test_accuracy": [accuracy_a1, accuracy_a2],
})
comparison_a


A-2はA-1よりパラメータ数が多いモデルです。しかし、活性化関数がない各隠れ層は線形変換なので、4層をまとめても1回の線形変換として表せます。

$$
W_4(W_3(W_2(W_1x+b_1)+b_2)+b_3)+b_4 = W'x+b'
$$

最後のSigmoidが0.5になる位置は、その直前の値が0になる位置です。したがって、モデルA-1もA-2も分類境界は直線になります。学習時のわずかな違いはあっても、層を増やしただけでは半月状データに適した曲線を表現できません。

#### モデルA-1とA-2の分類境界


In [ ]:
def plot_decision_boundary(ax, model, X, y, title):
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, 250),
        np.linspace(x2_min, x2_max, 250),
    )
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    probability = model.predict(grid, verbose=0).reshape(xx1.shape)

    ax.contourf(
        xx1, xx2, probability,
        levels=np.linspace(0, 1, 11), cmap="coolwarm", alpha=0.35,
    )
    ax.contour(
        xx1, xx2, probability,
        levels=[0.5], colors="black", linewidths=2,
    )
    ax.scatter(
        X[:, 0], X[:, 1],
        c=y, cmap="coolwarm", s=20, edgecolor="white", linewidth=0.3,
    )
    ax.set(xlabel="feature 1", ylabel="feature 2", title=title)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_decision_boundary(
    axes[0], linear_model_2, X_test_m, y_test_m,
    f"A-1: 2 hidden layers (accuracy={accuracy_a1:.3f})",
)
plot_decision_boundary(
    axes[1], linear_model_4, X_test_m, y_test_m,
    f"A-2: 4 hidden layers (accuracy={accuracy_a2:.3f})",
)
plt.tight_layout()
plt.show()


2つの図では、隠れ層を2層から4層へ増やしても、分類境界が直線のままであり、正解率も大きく変わらないことを確認します。

#### モデルB：ReLUあり・隠れ層2層


In [ ]:
keras.utils.set_random_seed(SEED)

relu_model = build_mlp(
    hidden_layer_count=2,
    hidden_activation="relu",
    model_name="relu_2_hidden_layers",
)

history_b, loss_b, accuracy_b = train_and_evaluate_mlp(
    relu_model
)

print(f"B（ReLUあり）のテスト正解率: {accuracy_b:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
plot_decision_boundary(
    ax, relu_model, X_test_m, y_test_m,
    f"B: ReLU, 2 hidden layers (accuracy={accuracy_b:.3f})",
)
plt.tight_layout()
plt.show()


モデルBでは、各隠れ層のReLUによって非線形な変換が加わります。そのため、分類境界を半月状データに合わせて曲げることができます。

#### 確認問題

1. モデルA-1とA-2では、パラメータ数、正解率、分類境界がどのように変わりましたか。
2. モデルA-2の隠れ層をさらに増やしても、曲線の分類境界を作れないのはなぜですか。
3. モデルBの分類境界が曲線になった理由を、ReLUの役割と結び付けて説明してください。
4. モデルBのReLUを`tanh`へ変更すると、分類境界と正解率はどう変わりますか。

> この演習の重要点は、パラメータや層を増やすだけでは表現できる境界の種類が変わらない場合があることです。**非線形な活性化関数が、MLPに曲がった分類境界を表現する能力を与えます。**

### 1-5. ニューラルネットワークはどう学習するか

学習は次の流れを繰り返します。

1. **順伝播**：入力から予測値を計算する
2. **損失の計算**：予測と正解のずれを数値化する
3. **誤差逆伝播**：各パラメータが損失に与えた影響を計算する
4. **最適化**：Adamなどを使って重みを更新する

![バックプロパゲーションと重み更新の流れ](images/session14_15/backpropagation.png)

#### バックプロパゲーションの役割

**バックプロパゲーション（誤差逆伝播法）**は、損失を小さくするには各重みをどちらへ、どの程度変えるべきかを調べる方法です。出力層で計算した損失の影響を、連鎖律を使って後ろの層から前の層へ伝えます。

1つの重み $w$ に対する勾配は、概念的には次のように計算されます。

$$
\frac{\partial L}{\partial w}
= \frac{\partial L}{\partial a}
  \frac{\partial a}{\partial z}
  \frac{\partial z}{\partial w}
$$

- 勾配が正：$w$を大きくすると損失が増える方向
- 勾配が負：$w$を大きくすると損失が減る方向
- 勾配の絶対値：その重みが損失に与える影響の大きさ

重要なのは、バックプロパゲーションとオプティマイザの役割を区別することです。

| 処理 | 役割 |
| --- | --- |
| バックプロパゲーション | 各パラメータについて損失の勾配を計算する |
| オプティマイザ | 計算された勾配を使ってパラメータを更新する |

単純な勾配降下法では、学習率を $\eta$ として次のように重みを更新します。

$$
w_{\mathrm{new}} = w_{\mathrm{old}} - \eta \frac{\partial L}{\partial w}
$$

学習率が小さすぎると学習に時間がかかり、大きすぎると最適な値を飛び越えて損失が安定しないことがあります。今回使用するAdamは、パラメータごとに更新幅を調整する代表的なオプティマイザです。

#### 確認問題

1. バックプロパゲーションとオプティマイザは、それぞれ何を担当しますか。
2. 勾配が正の場合、勾配降下法では重みを増やしますか、減らしますか。
3. 学習率が大きすぎる場合、損失はどのような変化をする可能性がありますか。
4. 活性化関数が微分可能、またはほぼすべての点で微分可能であることが重要なのはなぜですか。

画像分類も文章生成も、この基本的な流れは同じです。ただし、入力の構造と、特徴を抽出するための層が異なります。

---

## 2. CNN（Convolutional Neural Network）

### 2-1. 画像は数値の配列

グレースケール画像は「高さ × 幅 × 1」、カラー画像は通常「高さ × 幅 × 3」のテンソルとして表現されます。MNISTの画像は `28 × 28` ピクセルのグレースケール画像です。

画像を単純な全結合層へ入れるには、すべてのピクセルを1列に並べる必要があります。しかし、それでは「隣り合うピクセル」という画像の空間構造を直接利用しにくくなります。

### 2-2. フィルタで模様を探す

CNNの畳み込み層は、小さなフィルタを画像上で少しずつ動かし、似た模様がどこにあるかを調べます。まずは0と1だけの単純な画像で考えます。

次の6×6画像には、縦線があります。`1`が線、`0`が背景です。

```text
入力画像（6×6）

0 0 1 0 0 0
0 0 1 0 0 0
0 0 1 0 0 0
0 0 1 0 0 0
0 0 1 0 0 0
0 0 1 0 0 0
```

この画像に、次の2種類の3×3フィルタを適用します。

```text
縦線フィルタ        横線フィルタ

0 1 0              0 0 0
0 1 0              1 1 1
0 1 0              0 0 0
```

フィルタと画像の3×3領域を重ね、同じ位置の数字を掛けて合計します。

```text
縦線に縦線フィルタを重ねる

画像の小領域       フィルタ           掛け算の結果

0 1 0              0 1 0              0 1 0
0 1 0       ×      0 1 0       →      0 1 0
0 1 0              0 1 0              0 1 0

合計：3（強く反応）
```

同じ縦線に横線フィルタを重ねると、一致する位置は1か所だけです。

```text
縦線に横線フィルタを重ねる

画像の小領域       フィルタ           掛け算の結果

0 1 0              0 0 0              0 0 0
0 1 0       ×      1 1 1       →      0 1 0
0 1 0              0 0 0              0 0 0

合計：1（弱く反応）
```

つまり、フィルタが探している模様に近いほど、大きな値が次の層へ送られます。

| 画像の模様 | 縦線フィルタの反応 | 横線フィルタの反応 |
| --- | ---: | ---: |
| 縦線 | 3（強い） | 1（弱い） |
| 横線 | 1（弱い） | 3（強い） |
| 線がない | 0 | 0 |

実際のCNNでは、反応を必ず0か1にするわけではありません。反応の強さを数値として次の層へ渡します。また、フィルタの模様は人間が指定するのではなく、バックプロパゲーションによって学習されます。

### 2-3. フィルタを画像全体で動かす

3×3フィルタを6×6画像の端から端まで動かします。今回は画像の外側を埋めないため、フィルタを置ける位置は縦4か所、横4か所です。その結果、4×4の**特徴マップ**ができます。

```text
縦線フィルタで得られる特徴マップ（4×4）

0 3 0 0
0 3 0 0
0 3 0 0
0 3 0 0
```

値が3の位置は、「この付近に縦線らしい模様がある」ことを表します。同じ入力へ横線フィルタを使うと、次のように弱い反応になります。

```text
横線フィルタで得られる特徴マップ（4×4）

1 1 1 0
1 1 1 0
1 1 1 0
1 1 1 0
```

![畳み込みフィルタから特徴マップを作る流れ](images/session14_15/convolution.png)

`Conv2D(32, ...)`では32種類のフィルタを学習し、縦線、横線、斜め線、角などに反応する32枚の特徴マップを作ります。

CNNが画像に適している理由は次の2点です。

- **局所受容野**：近くのピクセルをまとめて調べられる
- **重み共有**：同じフィルタを画像のすべての位置で使うため、同じ模様をどこでも探せる

### 2-4. Max Poolingで情報を圧縮する

Max Poolingは、小領域の中で最も大きな反応だけを残します。先ほどの4×4特徴マップを2×2ずつ区切り、各領域の最大値を残すと、2×2になります。

```text
畳み込み後（4×4）             Max Pooling後（2×2）

0 3 │ 0 0                     3 0
0 3 │ 0 0       ─────→        3 0
────┼────
0 3 │ 0 0
0 3 │ 0 0
```

情報量の変化を数えると、次のようになります。

```text
入力画像             6×6 = 36個の値
  ↓ 畳み込み
特徴マップ           4×4 = 16個の値
  ↓ Max Pooling
圧縮後の特徴         2×2 =  4個の値
```

単に値を減らしているのではありません。細かなピクセル情報を、「どの模様が」「おおよそどこにあるか」という分類に役立つ情報へ置き換えています。

- 計算量を減らす
- 小さな位置ずれの影響を受けにくくする
- 強く反応した重要な特徴を残す

ただし、圧縮しすぎると細かな位置情報を失います。また、実際のCNNでは空間サイズを小さくする一方でフィルタ数を増やすことがあります。そのため、すべての数値が常に減るというより、**細かな画像を、より小さく抽象的な特徴表現へ変換する**と考えます。

### 2-5. MNISTを分類するCNNの階層構造

#### MNISTとは

**MNIST（Modified National Institute of Standards and Technology database）**は、0〜9の手書き数字画像を集めた、画像分類の代表的な学習用データセットです。

| 項目 | 内容 |
| --- | --- |
| 分類するもの | 手書き数字の0〜9 |
| クラス数 | 10クラス |
| 画像サイズ | 28×28ピクセル |
| 色 | グレースケール1チャンネル |
| ピクセル値 | 0〜255 |
| 学習用画像 | 60,000枚 |
| テスト用画像 | 10,000枚 |

```text
28×28の手書き数字画像
          ↓ CNN
0, 1, 2, 3, 4, 5, 6, 7, 8, 9
のどれであるかを予測
```

MNISTでは、背景に近いピクセルは0、白い筆跡に近いピクセルは255で表されます。モデルへ入力するときは、後の実習でピクセル値を0〜1へ変換します。

MNISTは画像が小さく、クラスも明確なので、CNNの畳み込み、プーリング、学習、評価を確認する入門用データとして適しています。ただし、現実の写真や医療画像より単純なため、MNISTで高い正解率が得られても、複雑な画像で同じ性能が得られるとは限りません。

CNNでは、一般に浅い層から深い層へ進むにつれて、特徴が抽象化されます。

```text
画像 → 線・エッジ → 角・曲線 → 部品 → 物体 → クラス確率
```

今回のモデルは、この28×28×1のMNIST画像を入力し、0〜9の10クラスへ分類します。層構成はKeras公式の[Simple MNIST convnet](https://keras.io/examples/vision/mnist_convnet/)と同じ基本構成です。畳み込みでは画像の外側を埋めない`padding="valid"`を使うため、畳み込みでも空間サイズが小さくなります。

```text
28×28×1
  ↓ Conv2D（32フィルタ、3×3、valid）
26×26×32
  ↓ MaxPooling2D
13×13×32
  ↓ Conv2D（64フィルタ、3×3、valid）
11×11×64
  ↓ MaxPooling2D
 5×5×64
  ↓ Flatten
1,600個の特徴
  ↓ Dropout(0.5)
  ↓ Dense(10, softmax)
数字0〜9の確率
```

1枚の特徴マップ内の空間位置は、`28×28 → 26×26 → 13×13 → 11×11 → 5×5`と小さくなります。フィルタ数は増えますが、元画像の細かな位置情報は段階的に圧縮され、最後は1,600個の特徴から10クラスの確率へ直接変換されます。

| 層の出力 | 空間サイズ | チャンネル数 | 値の総数 |
| --- | ---: | ---: | ---: |
| 入力 | 28×28 | 1 | 784 |
| 1回目のConv2D | 26×26 | 32 | 21,632 |
| 1回目のMax Pooling | 13×13 | 32 | 5,408 |
| 2回目のConv2D | 11×11 | 64 | 7,744 |
| 2回目のMax Pooling | 5×5 | 64 | 1,600 |
| Flatten | — | — | 1,600 |
| Dense出力 | — | — | 10 |

#### パラメータ数を抑える

畳み込み後に大きなDense層を置かず、1,600個の特徴から10クラスへ直接接続するため、モデル全体のパラメータ数は34,826です。

| 層 | パラメータ数 |
| --- | ---: |
| Conv2D：1チャンネル → 32フィルタ | 320 |
| Conv2D：32チャンネル → 64フィルタ | 18,496 |
| Dense：1,600 → 10 | 16,010 |
| **合計** | **34,826** |

比較として、画像を1列に並べて`Dense(64) → Dense(10)`へ入れる単純なMLPは50,890パラメータです。

| モデル | 主な構成 | パラメータ数 |
| --- | --- | ---: |
| 単純なMLP | `784 → Dense(64) → Dense(10)` | 50,890 |
| 今回のCNN | `Conv(32) → Conv(64) → Dense(10)` | 34,826 |

今回のCNNは単純なMLPより少ないパラメータで、画像の隣接関係を保ちながら局所的な模様を探せます。これが重み共有を使うCNNの利点です。

### 2-6. 出力と損失関数

10クラス分類では、出力層に10個のユニットを置き、`softmax` で合計1の確率へ変換します。MNISTの正解ラベルは0〜9の整数なので、損失関数には `sparse_categorical_crossentropy` を使います。

> `categorical_crossentropy` は正解をone-hot表現にした場合に使います。ラベルの形式と損失関数を対応させることが重要です。

---

# 第15回：Transformerの基礎とMNIST CNN演習

## 3. Transformer

CNNとTransformerは、どちらも入力から特徴を取り出すニューラルネットワークのアーキテクチャです。CNNが小さなフィルタで近くの模様を探すのに対し、Transformerは入力中の要素同士の関係を調べます。

### 3-1. 入力をトークンへ分ける

文章を扱う場合、最初に文章を**トークン**へ分割します。トークンは文字、単語、単語の一部など、モデルが処理する単位です。

```text
文章：私は猫が好きです

トークンの例： [私] [は] [猫] [が] [好き] [です]
```

各トークンはIDへ変換され、さらに多数の数値からなる**埋め込みベクトル**へ変換されます。使われ方が似ているトークンは、学習によって関連する表現を持つようになります。

Transformerはトークンだけでは語順を区別できないため、「何番目のトークンか」という**位置情報**も加えます。

```text
トークンの意味を表す埋め込み ＋ 文中の位置情報
                         ↓
              Transformerへの入力
```

### 3-2. Self-Attention：入力全体の関係を調べる

Self-Attentionは、各トークンについて「ほかのどのトークンを、どの程度参考にするか」を計算します。

```text
文章：猫が箱に入った。それは狭かった。

「それ」を理解するときに注目する候補

猫    ：中程度
箱    ：強い
入った：弱い
狭い  ：強い
```

この例では、「それ」だけを見るより、「箱」や「狭い」との関係を調べる方が意味を捉えやすくなります。Attentionの値は0か1ではなく、どこをどの程度参考にするかを表す重みです。

Self-Attentionでは、各トークンから次の3種類の情報を作ります。

| 名前 | 直感的な役割 |
| --- | --- |
| Query | 自分がどのような情報を探しているか |
| Key | 自分がどのような情報を持っているかを示す目印 |
| Value | 実際に相手へ渡す情報 |

QueryとKeyから注目度を計算し、その注目度に応じてValueを集めます。複数の注目方法を並行して使う仕組みを**Multi-Head Attention**と呼びます。あるヘッドは主語と述語、別のヘッドは指示語と対象など、異なる関係を学習できます。

### 3-3. Transformerブロック

Transformerでは、Self-AttentionとFeed Forward Networkを組み合わせたブロックを複数層重ねます。

![Transformerがトークン列を処理する流れ](images/session14_15/transformer.png)

- **Self-Attention**：入力要素同士の関係を取り込む
- **Feed Forward Network**：各要素の表現を変換する
- **残差接続**：変換前の情報を後ろへ伝え、深いモデルを学習しやすくする
- **正規化**：値のスケールを整え、学習を安定させる

このブロックを重ねることで、浅い層では単純な関係、深い層では文章全体の意味に関わる複雑な関係を表現できるようになります。

### 3-4. Transformerは文章専用ではない

Transformerはもともと系列を扱うために作られましたが、現在は文章以外にも使われます。

| 入力 | Transformerへ渡す単位 | 代表的な用途 |
| --- | --- | --- |
| 文章 | トークン | 分類、翻訳、文章生成 |
| 画像 | 小さく分割した画像パッチ | 画像分類、物体認識 |
| 音声 | 短い時間区間の特徴 | 音声認識、音声生成 |

画像を小さなパッチへ分割してTransformerで処理するモデルを**Vision Transformer（ViT）**と呼びます。したがって「CNNは画像、Transformerは文章」と完全に分けることはできません。

### 3-5. TransformerとLLMの関係

Transformerはアーキテクチャの名前です。一方、**LLM（Large Language Model：大規模言語モデル）**は、主にTransformerを大量の文章と多数のパラメータで学習した言語モデルです。

```text
Transformerという設計
        ＋
大量の文章データ
        ＋
多数のパラメータと計算資源
        ↓
       LLM
```

生成型LLMは、それまでのトークンから次のトークンの確率を予測します。

```text
入力：「データサイエンスを学ぶ目的は」

次のトークン候補
「問題」：高い確率
「分析」：中程度の確率
「空」  ：低い確率
```

選ばれたトークンを入力の末尾へ追加し、次のトークンを再び予測します。この処理を繰り返して文章を生成します。

LLMは、事前学習、指示調整、ファインチューニング、RAGなどを組み合わせて利用されます。ただし、生成結果は確率に基づくため、流暢でも事実とは限りません。

### 3-6. LLM利用時の注意

- **ハルシネーション**：存在しない事実や出典を生成することがある
- **バイアス**：学習データの偏りが出力へ反映されることがある
- **プロンプト依存性**：指示の表現や文脈で出力が変わる
- **機密性**：個人情報、診療情報、未公開データを安易に入力しない
- **検証責任**：重要な判断では一次資料や専門家による確認が必要

---

## 4. CNNとTransformerを比較する

CNNとTransformerを比べると、どちらも「重要な特徴を取り出す」という目的は共通していますが、情報の集め方が異なります。

| 観点 | CNN | Transformer |
| --- | --- | --- |
| 入力の代表例 | 画像 | トークン列、画像パッチ、音声特徴 |
| 中心的な処理 | 畳み込み | Self-Attention |
| 最初に見る範囲 | 小さな局所領域 | 入力全体の要素関係 |
| 同じ処理の再利用 | 同じフィルタを各位置で使う | 同じAttention計算を各要素に使う |
| 位置の扱い | 画像の格子構造を直接利用する | 位置情報を入力へ加える |
| 特徴の作り方 | 線→形→部品のように階層化 | 関係を取り込み、文脈に応じて表現を更新 |
| 共通する学習 | 損失、バックプロパゲーション、オプティマイザによる更新 |

```text
CNN
近くの模様を調べる → 圧縮する → より複雑な模様を調べる

Transformer
入力全体の関係を調べる → 各要素の表現を更新する → 層を重ねる
```

### 確認問題

1. CNNの縦線フィルタは、縦線と横線のどちらに強く反応しますか。
2. Max Poolingで特徴マップを小さくする利点と欠点を説明してください。
3. Self-Attentionは、各トークンについて何を計算していますか。
4. CNNとTransformerでは、最初に情報を集める範囲がどのように異なりますか。
5. TransformerとLLMの関係を説明してください。
6. LLMの回答を重要な意思決定にそのまま使ってはいけない理由を2つ挙げてください。

---

## 5. 実行環境の準備

Google Colabでは、必要に応じて「ランタイム」→「ランタイムのタイプを変更」からGPUを選択できます。MNISTはCPUでも実行できます。

グラフのタイトルや軸ラベルに日本語を使用しても文字化けしないよう、前章と同じ`japanize-matplotlib`を設定します。ライブラリがない場合は、このセル内で自動的にインストールします。


In [ ]:
import random
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras

from keras import layers
from sklearn.metrics import confusion_matrix, classification_report

try:
    import japanize_matplotlib
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "japanize-matplotlib", "-q"]
    )
    import japanize_matplotlib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "IPAexGothic"
plt.rcParams["axes.unicode_minus"] = False

print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 6. MNISTデータを読み込む

MNISTには、0〜9の手書き数字画像が学習用60,000枚、テスト用10,000枚含まれます。


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test :", x_test.shape, x_test.dtype)
print("y_test :", y_test.shape, y_test.dtype)
print("pixel range:", x_train.min(), "〜", x_train.max())


### 演習1：データを観察する

次のセルを実行し、画像とラベルが対応していることを確認します。


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(10, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"label = {y_train[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


クラス数も確認します。


In [ ]:
class_counts = pd.Series(y_train).value_counts().sort_index()
display(class_counts.rename("count").to_frame())

class_counts.plot(kind="bar", figsize=(8, 4), color="steelblue")
plt.xlabel("digit")
plt.ylabel("count")
plt.title("MNIST training class distribution")
plt.show()


**考察**

- 画像の形状 `(28, 28)` は何を表していますか。
- 数字ごとの枚数は完全に同じですか。
- 背景と文字は、それぞれどのようなピクセル値ですか。

## 7. 前処理

ピクセル値を0〜255から0〜1へ変換し、チャンネル次元を追加します。


In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("x_train:", x_train.shape)
print("x_test :", x_test.shape)
print("pixel range:", x_train.min(), "〜", x_train.max())


> 形状の最後の `1` はグレースケールの1チャンネルを表します。RGB画像なら通常は `3` です。

### 演習2：前処理を説明する

次の問いに文章で答えてください。

1. 255で割る目的は何ですか。
2. `(60000, 28, 28)` を `(60000, 28, 28, 1)` に変えた理由は何ですか。
3. `y_train` をone-hot表現に変換していないのに使える損失関数は何ですか。

## 8. CNNモデルを構築する

### 8-1. Dropoutとdrop rate

ニューラルネットワークのユニット数や層数が増えると、学習データにだけ細かく適合し、未知データへの性能が下がる**過学習**が起きることがあります。Dropoutは、過学習を抑えるための正則化手法の1つです。

学習中に、層の出力の一部をランダムに0へ置き換えます。毎回異なるユニットが無効になるため、モデルは特定のユニットだけに頼りにくくなります。

![Dropoutでユニットを無効化する流れ](images/session14_15/dropout.png)

`layers.Dropout(0.5)`の`0.5`が**drop rate**です。

| 項目 | `Dropout(0.5)`での意味 |
| --- | --- |
| 無効にする割合 | 50% |
| 残す割合 | 50% |
| 1,600個の特徴のうち、1回の処理で残る数の期待値 | $1,600 \times 0.5 = 800$個 |

Dropoutの動作は学習時と評価・予測時で異なります。

| 状態 | Dropoutの動作 |
| --- | --- |
| 学習時：`model.fit()` | 指定割合をランダムに無効化する |
| 評価時：`model.evaluate()` | すべてのユニットを使用する |
| 予測時：`model.predict()` | すべてのユニットを使用する |

Kerasは、学習時と評価・予測時の切り替えや出力スケールの調整を自動的に行います。drop rateを大きくしすぎると必要な情報まで失われ、学習不足になる可能性があります。今回の`0.5`はKeras公式例に合わせた値ですが、固定的な正解ではなく、検証データを見ながら調整するハイパーパラメータです。

### 8-2. モデルの定義

ここから先は、必要な処理を自分で整理し、AIにコード作成を依頼しながら進めます。AIが生成したコードをそのまま実行するのではなく、指定した条件が反映されているか、前のセルで作成した変数とつながっているかを確認してください。

#### 作成するモデル

次の順に層を配置します。

| 順番 | 層 | 設定 | 目的 |
| ---: | --- | --- | --- |
| 1 | Input | `(28, 28, 1)` | MNIST画像を受け取る |
| 2 | Conv2D | 32フィルタ、3×3、valid、ReLU、名前は`conv1` | 局所特徴を抽出する |
| 3 | MaxPooling2D | 2×2 | 特徴マップを縮小する |
| 4 | Conv2D | 64フィルタ、3×3、valid、ReLU | より複雑な特徴を抽出する |
| 5 | MaxPooling2D | 2×2 | 特徴マップを縮小する |
| 6 | Flatten |  | 全結合層へ渡せる形にする |
| 7 | Dropout | drop rate 0.5 | 過学習を抑える |
| 8 | Dense | 10ユニット、Softmax | 0〜9の確率を出力する |

モデル名は`mnist_cnn`、変数名は`model`とします。作成後、Adam、`sparse_categorical_crossentropy`、正解率を指定してコンパイルし、`summary()`を表示します。

**AIへの依頼例**

```text
Keras 3のSequential APIを使って、MNIST分類用CNNを作成してください。
入力は28×28×1です。以下の順に層を配置してください。
1. 32枚の3×3 Conv2D。paddingはvalid、活性化関数はReLU、層名はconv1
2. 2×2 MaxPooling2D
3. 64枚の3×3 Conv2D。paddingはvalid、活性化関数はReLU
4. 2×2 MaxPooling2D
5. Flatten
6. drop rate 0.5のDropout
7. 10ユニットのDense。活性化関数はSoftmax
モデルの変数名はmodel、モデル名はmnist_cnnにしてください。
Adam、sparse_categorical_crossentropy、accuracyでコンパイルし、summaryも表示してください。
すでにimport kerasとfrom keras import layersは実行済みです。
```


In [ ]:
# AIが作成したCNNの定義、コンパイル、summary表示をここに記入する


#### AIが作ったコードの確認

- 入力形状の最後にチャンネル数`1`があるか
- 2つの畳み込み層のフィルタ数が32、64になっているか
- 2つの畳み込み層が`padding="valid"`になっているか
- 畳み込み層がReLU、出力層がSoftmaxになっているか
- ラベル形式に対応した損失関数になっているか
- 最初の畳み込み層に`conv1`という名前が付いているか
- `model.summary()`の総パラメータ数が34,826になっているか

### 演習3：モデルの形状とパラメータ

`model.summary()`を見て答えてください。

1. 1回目のMax Pooling後の形状はいくつですか。
2. 2回目のMax Pooling後の形状はいくつですか。
3. 最初のConv2D層のパラメータ数を、次式で確認してください。

$$
(\text{カーネル高} \times \text{カーネル幅} \times \text{入力チャンネル数} + 1) \times \text{フィルタ数}
$$

4. 最後のDense層のユニット数が10である理由を説明してください。
5. Dense層のパラメータ数が16,010になる理由を説明してください。
6. CNN全体のパラメータ数34,826は、単純なMLPの50,890よりいくつ少ないですか。
7. `Dropout(0.5)`は学習時に何%を無効化し、何%を残しますか。
8. Dropoutを評価・予測時にもランダムに適用すると、どのような問題が起こるか考えてください。

## 9. モデルを学習する

学習データの20%を検証データとして分けます。テストデータは、モデルの最終評価まで使用しません。

実装する条件は次のとおりです。

- エポック数：5
- バッチサイズ：128
- 検証データ：学習データの20%
- EarlyStoppingの監視対象：検証損失
- EarlyStoppingの`patience`：2
- 最も良かった重みを復元する
- 学習履歴の変数名：`history`
- テストデータはまだ使わない

**AIへの依頼例**

```text
コンパイル済みのKerasモデルmodelを学習するコードを書いてください。
x_trainとy_trainを使用し、エポック数5、バッチサイズ128、validation_split=0.2とします。
検証損失を監視するEarlyStoppingを設定し、patience=2、restore_best_weights=Trueにしてください。
学習履歴はhistoryへ保存してください。テストデータは使用しないでください。
```


In [ ]:
# AIが作成したEarlyStoppingとモデル学習のコードをここに記入する


> `x_test`や`y_test`が`fit()`の中に入っていた場合は修正してください。テストデータを見ながらモデルを調整すると、最終評価が楽観的になります。

### 学習曲線を確認する

`history.history`には、エポックごとの学習損失、検証損失、学習正解率、検証正解率が記録されています。これをDataFrameに変換し、損失と正解率を左右に並べて可視化します。

**AIへの依頼例**

```text
Kerasの学習履歴historyについて、history.historyをpandasのDataFrameへ変換し、表として表示してください。
次にmatplotlibで1行2列のグラフを作ってください。
左は学習損失lossと検証損失val_loss、右は学習正解率accuracyと検証正解率val_accuracyです。
横軸は1から始まるepochとし、凡例、軸ラベル、タイトルを付けてください。
```


In [ ]:
# AIが作成した学習履歴の表と学習曲線のコードをここに記入する


### 演習4：学習曲線を読む

- エポックが進むと、学習・検証の損失はどう変化しましたか。
- 学習精度だけが上がり、検証精度が改善しなくなる現象を何と呼びますか。
- `Dropout` と `EarlyStopping` はそれぞれ何のために使っていますか。

## 10. テストデータで最終評価する

学習とモデル選択が終わったので、ここで初めてテストデータを使います。`model.evaluate()`でテスト損失とテスト正解率を計算し、小数第4位まで表示します。変数名は`test_loss`と`test_accuracy`にします。

**AIへの依頼例**

```text
学習済みKerasモデルmodelをx_testとy_testで評価してください。
返された損失をtest_loss、正解率をtest_accuracyへ保存し、小数第4位まで表示するコードを書いてください。
```


In [ ]:
# AIが作成したテストデータでの最終評価コードをここに記入する


正解率だけでは、どの数字を間違えやすいか分かりません。予測クラスを求め、混同行列とクラス別指標を確認します。

次に、各画像について10クラスの予測確率を求め、最も確率が高い数字を予測クラスとします。その後、混同行列をヒートマップで表示し、クラス別のprecision、recall、F1-scoreを確認します。

使用する変数名：

- `y_prob`：10クラスの予測確率
- `y_pred`：予測クラス
- `cm`：混同行列

**AIへの依頼例**

```text
学習済みKerasモデルmodelでx_testの予測確率を計算してy_probへ保存してください。
各行で最も確率が高いクラスをNumPyで求め、y_predへ保存してください。
y_testとy_predから混同行列cmを作り、seabornのheatmapで数字も表示してください。
横軸をpredicted label、縦軸をtrue labelとしてください。
最後にscikit-learnのclassification_reportを小数第4位まで表示してください。
必要なライブラリはすでに読み込み済みです。
```


In [ ]:
# AIが作成した予測、混同行列、クラス別評価のコードをここに記入する


#### AIが作ったコードの確認

- `argmax`を取る軸がクラス方向になっているか
- 混同行列で正解と予測の順番が逆になっていないか
- 混同行列の縦軸と横軸の表示が計算内容と一致しているか

### 演習5：評価結果を読む

1. テスト正解率はいくつでしたか。
2. 混同行列の対角成分は何を表しますか。
3. 最も多い誤分類の「正解→予測」の組み合わせを調べてください。

対角成分は正しく分類した数なので、コピーした混同行列の対角成分を0にしてから最大値の位置を求めます。元の`cm`は後でも使用するため、直接書き換えないでください。

**AIへの依頼例**

```text
NumPy配列cmは、行が正解ラベル、列が予測ラベルの混同行列です。
cmをコピーし、コピーした配列の対角成分を0にして、対角成分以外で最大となる行番号、列番号、件数を求めて表示してください。
元のcmは変更しないでください。
```


In [ ]:
# AIが作成した「最も多い誤分類」を求めるコードをここに記入する


## 11. 誤分類画像を観察する

誤分類された画像のインデックスを求め、先頭15枚を3行5列で表示します。各画像には正解、予測、予測クラスに対する確信度を表示します。

**AIへの依頼例**

```text
y_testとy_predが異なる要素のインデックスをNumPyで求めてwrong_indicesへ保存してください。
誤分類画像の先頭15枚を、matplotlibで3行5列に表示してください。
画像はx_testから取り出し、グレースケールで表示します。
各画像のタイトルに正解ラベル、予測ラベル、モデルが予測クラスへ与えた確率を小数第2位まで表示してください。
予測確率はy_probに保存されています。軸は非表示にしてください。
```


In [ ]:
# AIが作成した誤分類画像の可視化コードをここに記入する


### 演習6：誤分類を考察する

誤分類画像を3枚以上選び、次の観点で考察してください。

- 人間にとっても判別しにくい書き方か
- どの数字の特徴と似ているか
- モデルの確信度は高いか低いか
- 画像のずれ、線の太さ、欠けなどが影響していそうか

## 12. 畳み込み層の特徴マップを見る

1枚の画像が、最初の畳み込み層によってどのような特徴マップへ変換されるかを確認します。

モデル作成時に最初の畳み込み層へ`conv1`という名前を付けました。この層の出力を取り出す中間モデルを作成し、テスト画像1枚から得られる32枚の特徴マップを4行8列で表示します。

**AIへの依頼例**

```text
学習済みKerasモデルmodelから、名前がconv1の層の出力を取得する中間モデルconv1_modelを作ってください。
x_testの0番目の画像1枚を入力し、特徴マップをfeature_mapsへ保存してください。
conv1には32フィルタがあるので、32枚の特徴マップをmatplotlibで4行8列に表示してください。
各図にはチャンネル番号を付け、軸は非表示にしてください。
全体タイトルには元画像の正解ラベルも表示してください。
```


In [ ]:
# AIが作成した中間モデルと特徴マップの可視化コードをここに記入する


#### AIが作ったコードの確認

- 中間モデルへの入力が元の`model.inputs`になっているか
- 出力が最終分類結果ではなく`conv1`層の出力になっているか
- 画像1枚でもバッチ次元を残して入力しているか
- 特徴マップのチャンネル方向を32枚に分けて表示しているか

> 特徴マップの意味は、1枚だけを見て断定できません。「縦線に反応しているように見える」など、観察に基づく表現を使いましょう。

## 13. 発展課題：モデルを改善して比較する

基準モデルから**1項目だけ**変更し、同じ条件で再学習してください。複数項目を同時に変えると、どの変更が結果に影響したか判断しにくくなります。

変更候補：

- 1層目のフィルタ数：`32 → 16` または `64`
- カーネルサイズ：`3 → 5`
- Dropout率：`0.5 → 0.3` または `0.7`
- 畳み込みのpadding：`valid → same`
- Batch size：`128 → 64` または `256`
- データ拡張：小さな回転や平行移動を追加

変更する項目を1つ決めてから、AIへ基準モデルの条件と変更点を伝えます。変数名は`improved_model`とし、基準モデルと同じデータ分割、エポック数、評価方法を使用してください。

**AIへの依頼例**

```text
先ほど作成したMNIST用CNNを基準に、Dropout率だけを0.5から0.3へ変更したKerasモデルを作ってください。
変数名はimproved_modelとしてください。他の層構成、活性化関数、optimizer、loss、metricsは基準モデルと同じにしてください。
基準モデルと同じ条件で学習・評価し、パラメータ数、最高検証精度、テスト精度を取得してください。
```


In [ ]:
# 変更点を1つ決め、AIが作成した改良モデルの定義・学習・評価を記入する


#### 比較時の注意

- 基準モデルから変更した項目を明記する
- テストデータを学習やEarlyStoppingに使わない
- 乱数シード、検証割合、エポック数など、変更対象以外の条件を揃える
- 1回の結果だけで「必ず改善する」と結論付けない

結果を次の表へ記録してください。

| モデル | 変更点 | パラメータ数 | 最高検証精度 | テスト精度 | 考察 |
| --- | --- | ---: | ---: | ---: | --- |
| 基準CNN | なし |  |  |  |  |
| 改良CNN |  |  |  |  |  |

---

> 実行のたびに値がわずかに変わることがあります。乱数シードを固定しても、GPUなどの実行環境によって完全には一致しない場合があります。

---

## 14. まとめ

- CNNは、フィルタで局所的な模様を探し、プーリングで重要な反応を残しながら空間情報を圧縮する
- Transformerは、Self-Attentionによって入力要素同士の関係を取り込み、各要素の表現を更新する
- CNNとTransformerは情報の集め方が異なるが、損失を小さくするようにパラメータを学習する点は共通する
- LLMは、主にTransformerを大量の文章データと多数のパラメータで学習した言語モデルである
- モデル評価では、正解率だけでなく学習曲線、混同行列、個別の誤分類例も確認する
- 高い性能が得られても、未知データへの一般化、バイアス、説明可能性、利用目的を検討する必要がある

---

**Last updated**: 2026-07-20  
**Instructor**: Nakaura-T (DS Department, Kumamoto Univ)
